# 02 — Toy Diffusion Model (DDPM)

**Module 2 · Week 4 · TA-2**

This notebook covers:
- Forward process: adding noise to data
- Noise schedule (linear beta schedule)
- Denoiser network: predicting noise from noisy input + timestep
- Training loop
- Sampling (reverse diffusion)

We use a simple 2D toy distribution (Swiss roll or Gaussian mixture) to keep compute light.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_swiss_roll

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
T = 200  # number of diffusion timesteps (small for quick demo)
print(f'Device: {device}')

## 1. Generate Toy Dataset

In [ ]:
# 2D Swiss roll (use first 2 coordinates)
data_np, _ = make_swiss_roll(n_samples=10000, noise=0.1)
data_np = data_np[:, [0, 2]]  # keep x and z
data_np = (data_np - data_np.mean(0)) / data_np.std(0)  # normalize

data = torch.tensor(data_np, dtype=torch.float32)
print(f'Data shape: {data.shape}')

plt.figure(figsize=(5, 5))
plt.scatter(data[:, 0], data[:, 1], s=1, alpha=0.3)
plt.title('Target distribution (2D Swiss Roll)')
plt.show()

## 2. Noise Schedule

In [ ]:
# Linear beta schedule
def make_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02):
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alpha_bar = torch.cumprod(alphas, dim=0)  # cumulative product
    return betas, alphas, alpha_bar

betas, alphas, alpha_bar = make_beta_schedule(T)
betas = betas.to(device)
alpha_bar = alpha_bar.to(device)

# Visualize noise schedule
plt.plot(alpha_bar.cpu(), label='alpha_bar (signal retention)')
plt.xlabel('Timestep t')
plt.ylabel('alpha_bar')
plt.title('Noise schedule: signal fades to zero')
plt.legend()
plt.show()

## 3. Forward Process: Add Noise

In [ ]:
def q_sample(x0, t, alpha_bar):
    """Forward process: sample x_t given x_0 at timestep t."""
    # --- TODO: Implement forward process ---
    # x_t = sqrt(alpha_bar[t]) * x0 + sqrt(1 - alpha_bar[t]) * noise
    # Return x_t and the noise
    pass

# Visualize forward process at different timesteps
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
x0 = data[:500].to(device)
for i, t_val in enumerate([0, 50, 100, 150, 199]):
    t = torch.full((500,), t_val, dtype=torch.long, device=device)
    x_t, _ = q_sample(x0, t, alpha_bar)
    axes[i].scatter(x_t[:, 0].cpu(), x_t[:, 1].cpu(), s=1, alpha=0.3)
    axes[i].set_title(f't = {t_val}')
plt.suptitle('Forward process: data → noise')
plt.tight_layout()
plt.show()

## 4. Denoiser Network

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Encode timestep t as a sinusoidal embedding vector."""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class Denoiser(nn.Module):
    """MLP denoiser: predicts noise given (x_t, t)."""
    def __init__(self, data_dim: int = 2, time_dim: int = 32, hidden: int = 128):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbedding(time_dim)
        # --- TODO: Define MLP ---
        # Input: data_dim + time_dim
        # Hidden: 3 layers of `hidden` units with SiLU activation
        # Output: data_dim (predicted noise)
        pass

    def forward(self, x, t):
        t_emb = self.time_emb(t)
        # --- TODO: Concatenate x and t_emb, pass through MLP ---
        pass


denoiser = Denoiser().to(device)
print(f'Denoiser parameters: {sum(p.numel() for p in denoiser.parameters()):,}')

## 5. Training Loop

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(data)
loader = DataLoader(dataset, batch_size=256, shuffle=True)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=3e-4)

losses = []
NUM_EPOCHS = 50

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    for (x0_batch,) in loader:
        x0_batch = x0_batch.to(device)
        B = x0_batch.size(0)

        # --- TODO: Training step ---
        # 1. Sample random timestep t for each sample in batch
        # 2. Compute noisy x_t and true noise using q_sample
        # 3. Predict noise with denoiser(x_t, t)
        # 4. Loss = MSE(predicted_noise, true_noise)
        # 5. Backprop and step

    losses.append(epoch_loss / len(loader))
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}  Loss: {losses[-1]:.4f}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss')
plt.show()

## 6. Sampling (Reverse Diffusion)

In [ ]:
@torch.no_grad()
def ddpm_sample(denoiser, n_samples: int, T: int, betas, alpha_bar):
    """Generate samples by running the reverse diffusion process."""
    alphas = 1.0 - betas
    # Start from pure noise
    x = torch.randn(n_samples, 2, device=device)

    for t_val in reversed(range(T)):
        t = torch.full((n_samples,), t_val, dtype=torch.long, device=device)
        # --- TODO: Implement DDPM reverse step ---
        # predicted_noise = denoiser(x, t)
        # x = (1/sqrt(alpha_t)) * (x - beta_t/sqrt(1-alpha_bar_t) * predicted_noise)
        # + sigma_t * z   (where z ~ N(0,I), skip for t=0)
        pass

    return x

samples = ddpm_sample(denoiser, n_samples=1000, T=T, betas=betas, alpha_bar=alpha_bar)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(data[:1000, 0], data[:1000, 1], s=2, alpha=0.3)
axes[0].set_title('Real data')
axes[1].scatter(samples[:, 0].cpu(), samples[:, 1].cpu(), s=2, alpha=0.3)
axes[1].set_title('Generated samples')
plt.tight_layout()
plt.show()